# Day 19 – Prompt Optimisation – Extensive Solutions

Complete solutions using DSPy, and a simple gradient‑based AutoPrompt example.

In [ ]:
# !pip install dspy-ai
import dspy
from dspy.teleprompt import BootstrapFewShot, LabeledFewShot
import openai
from dotenv import load_dotenv
import os
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

# Set up DSPy with OpenAI
lm = dspy.OpenAI(model="gpt-3.5-turbo")
dspy.settings.configure(lm=lm)

print("DSPy ready.")

## Exercise 1: Optimise a prompt for sentiment classification

We'll define a signature, create a training set, and use BootstrapFewShot.

In [ ]:
class Sentiment(dspy.Signature):
    """Classify the sentiment of a text as positive or negative."""
    text = dspy.InputField()
    sentiment = dspy.OutputField(desc="positive or negative")

# Create a predictor
predictor = dspy.Predict(Sentiment)

# Training examples
trainset = [
    dspy.Example(text="I love this product!", sentiment="positive"),
    dspy.Example(text="This is terrible.", sentiment="negative"),
    dspy.Example(text="Amazing experience!", sentiment="positive"),
    dspy.Example(text="Waste of money.", sentiment="negative"),
]

# Define a simple metric
def metric(example, pred, trace=None):
    return example.sentiment == pred.sentiment

# Optimise with few-shot
optimizer = BootstrapFewShot(metric=metric, max_bootstrapped_demos=2)
optimized_predictor = optimizer.compile(predictor, trainset=trainset)

# Test
test_text = "Absolutely fantastic!"
result = optimized_predictor(text=test_text)
print(f"Text: {test_text}\nPredicted sentiment: {result.sentiment}")

# Inspect the prompt (optional, DSPy doesn't expose easily but you can print the predictor)
print(optimized_predictor.demos)  # Shows few-shot examples

## Exercise 2: Use DSPy with a local model (Ollama)

We can use DSPy's `OllamaLocal` client.

In [ ]:
# local_lm = dspy.OllamaLocal(model="llama3", max_tokens=100)
# dspy.settings.configure(lm=local_lm)
# Then same as above
print("To use local model, install Ollama and set dspy.OllamaLocal.")

## Exercise 3: Compare OPRO (via OpenAI) with DSPy

OPRO (Optimising Prompts by Reinforcement Learning) is not directly in DSPy, but we can simulate a simple iterative prompt refinement.

In [ ]:
def opro_style_optimize(base_prompt, trainset, eval_func, iterations=3):
    best_prompt = base_prompt
    best_score = 0
    for i in range(iterations):
        # Evaluate current prompt
        score = eval_func(best_prompt, trainset)
        if score > best_score:
            best_score = score
        # Ask LLM to improve the prompt
        improvement_prompt = f"""The current prompt is: "{best_prompt}"
        It achieved an accuracy of {score:.2f} on the training set.
        Suggest a better version of the prompt to improve accuracy.
        Output only the new prompt.
        """
        new_prompt = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": improvement_prompt}],
            temperature=0.7
        ).choices[0].message.content
        best_prompt = new_prompt
    return best_prompt

# Example usage (pseudo)
# def eval_func(prompt, data): ...
# best = opro_style_optimize("Classify sentiment.", trainset, eval_func)
print("OPRO simulation – run with actual eval function.")

## Exercise 4: Gradient‑based AutoPrompt (simplified for understanding)

AutoPrompt searches for trigger tokens by gradient. We'll show a simplified version using `transformers` and `torch`.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

# This is a simplified illustration, not a full implementation
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# We want to find a trigger that makes the model output "positive" for a given input
# In practice, you would optimise the trigger tokens directly.
print("Full AutoPrompt implementation is complex; see original paper.")
print("Reference: https://arxiv.org/abs/2010.15980")